# 🧠 Brain Tumor Detection: GPU Training on Google Colab
### Fast Training for ResNet-18, MobileNetV3, and Vision Transformer (ViT-B/16)

This notebook is tailored to train the computationally intensive models on **Google Colab's Free GPU (T4)**, and export the trained `.pth` checkpoint files back to your local project's `models/` directory.

---

### ⚙️ Step 1: Switch Runtime to GPU
In Google Colab, go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU not detected! Make sure to select T4 GPU in Runtime settings.")

## 📦 Step 2: Install Required Libraries

In [ ]:
!pip install -q scikit-learn matplotlib seaborn tqdm

## 📂 Step 3: Dataset Setup on Colab
Choose **Option A** (Download directly from Kaggle) or **Option B** (Mount Google Drive if you uploaded your dataset zip).

In [ ]:
# OPTION A: Download directly from Kaggle (Fastest)
# If you have kaggle.json, place it in /root/.kaggle/kaggle.json, then:
# !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p dataset/

# OPTION B: Mount Google Drive if your dataset is in Drive
from google.colab import drive
import os
import shutil
from pathlib import Path

# Mount drive
drive.mount('/content/drive')

# Set DATASET_PATH to where your brain-tumor-mri-dataset folder is located
# If you uploaded a zip, e.g. /content/drive/MyDrive/brain-tumor-mri-dataset.zip:
# !unzip -q /content/drive/MyDrive/brain-tumor-mri-dataset.zip -d /content/dataset/

# Let's verify dataset directories:
POSSIBLE_PATHS = [
    Path('/content/dataset/brain-tumor-mri-dataset'),
    Path('/content/dataset'),
    Path('/content/drive/MyDrive/brain-tumor-mri-dataset'),
    Path('/content/brain-tumor-mri-dataset')
]

DATA_DIR = None
for p in POSSIBLE_PATHS:
    if (p / 'Training').exists() and (p / 'Testing').exists():
        DATA_DIR = p
        break

if DATA_DIR:
    print(f"✅ Dataset found at: {DATA_DIR}")
    print("Classes in Training:", os.listdir(DATA_DIR / 'Training'))
else:
    print("⚠️ Please adjust DATA_DIR to match your dataset location.")

## 🏗️ Step 4: Model Architectures & Data Pipeline
These match your local project exactly so that saved `.pth` files are 100% drop-in compatible.

In [ ]:
import os
import time
import random
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# Configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def is_valid_image(p_str):
    try:
        p = Path(p_str)
        return p.suffix.lower() in {".jpg", ".jpeg", ".png"} and p.stat().st_size > 500
    except:
        return False

# Model Builders
def build_resnet18(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes),
    )
    return model

def build_mobilenet_v3(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v3_small(weights=weights)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.Hardswish(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )
    return model

def build_vit_b16(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.vit_b_16(weights=weights)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if not name.startswith("heads"):
                param.requires_grad = False
    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )
    return model

print("✅ Pipeline components and architectures defined.")

## 🏋️ Step 5: Training Function (Fast GPU Execution)

In [ ]:
def train_model(model_name, build_fn, epochs=10, lr=1e-4, batch_size=32, freeze_backbone=False):
    print(f"\n{'='*65}")
    print(f"🚀 Training {model_name} on {DEVICE} (Epochs: {epochs}, Batch: {batch_size}, LR: {lr})")
    print(f"{'='*65}")
    
    train_dir = DATA_DIR / 'Training'
    full_dataset = datasets.ImageFolder(root=str(train_dir), is_valid_file=is_valid_image)
    
    val_size = int(len(full_dataset) * 0.15)
    train_size = len(full_dataset) - val_size
    train_sub, val_sub = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
    
    class TransformDataset(torch.utils.data.Dataset):
        def __init__(self, subset, transform):
            self.subset = subset
            self.transform = transform
        def __getitem__(self, idx):
            x, y = self.subset[idx]
            return self.transform(x), y
        def __len__(self):
            return len(self.subset)
            
    train_loader = DataLoader(TransformDataset(train_sub, train_transforms), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(TransformDataset(val_sub, test_transforms), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    model = build_fn(num_classes=4, pretrained=True, freeze_backbone=freeze_backbone).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    output_pth = f"models/{model_name}_final_model.pth"
    
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [Train]", leave=False)
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss.backward()
            optimizer.step()
            
            tr_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            tr_correct += (preds == lbls).sum().item()
            tr_total += imgs.size(0)
            pbar.set_postfix({'loss': f"{tr_loss/tr_total:.4f}", 'acc': f"{tr_correct/tr_total*100:.1f}%"})
            
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                logits = model(imgs)
                loss = criterion(logits, lbls)
                val_loss += loss.item() * imgs.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == lbls).sum().item()
                val_total += imgs.size(0)
                
        tr_acc = tr_correct / tr_total
        val_acc = val_correct / val_total
        val_loss_avg = val_loss / val_total
        scheduler.step(val_loss_avg)
        
        history['train_loss'].append(round(tr_loss / tr_total, 4))
        history['train_acc'].append(round(tr_acc * 100, 2))
        history['val_loss'].append(round(val_loss_avg, 4))
        history['val_acc'].append(round(val_acc * 100, 2))
        
        saved = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'model_state_dict': model.state_dict(),
                'class_names': CLASS_NAMES,
                'val_acc': val_acc
            }, output_pth)
            saved = " -> ⭐ Best Checkpoint Saved"
            
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {tr_loss/tr_total:.4f} (Acc: {tr_acc*100:.2f}%) | Val Loss: {val_loss_avg:.4f} (Acc: {val_acc*100:.2f}%){saved}")
        
    with open(f"results/{model_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    print(f"\n✅ Finished {model_name}. Best Val Acc: {best_val_acc*100:.2f}%. Saved to {output_pth}")
    return model, history

## 🚀 Step 6: Train ResNet-18, MobileNetV3, and Vision Transformer (ViT)

In [ ]:
# 1. Train ResNet-18 (Takes ~1-2 minutes on T4 GPU)
resnet_model, resnet_hist = train_model('resnet18', build_resnet18, epochs=10, lr=1e-4, batch_size=32)

In [ ]:
# 2. Train MobileNetV3 (Takes ~1-2 minutes on T4 GPU)
mobilenet_model, mobilenet_hist = train_model('mobilenet_v3', build_mobilenet_v3, epochs=10, lr=1e-4, batch_size=32)

In [ ]:
# 3. Train Vision Transformer (ViT-B/16) (Takes ~4-5 minutes on T4 GPU)
# For ViT, name the output checkpoint best_vit_model.pth
vit_model, vit_hist = train_model('vit', build_vit_b16, epochs=10, lr=1e-4, batch_size=32)
if os.path.exists('models/vit_final_model.pth'):
    shutil.copy('models/vit_final_model.pth', 'models/best_vit_model.pth')

## 💾 Step 7: Download the Trained Checkpoints to Your Computer
Download the `.pth` files and place them into your local project's `models/` folder.

In [ ]:
from google.colab import files

# Zip the models and results for easy download
!zip -r trained_models_and_histories.zip models/ results/

# Trigger download in browser
files.download('trained_models_and_histories.zip')

### 🏁 What to do after downloading:
1. Unzip `trained_models_and_histories.zip` on your computer.
2. Place the `.pth` files into `c:\Users\Parth\OneDrive\Documents\AIML Project\models\`:
   - `resnet18_final_model.pth`
   - `mobilenet_v3_final_model.pth`
   - `best_vit_model.pth`
   (Your `custom_cnn_final_model.pth` is already trained locally!)
3. Place the `*_history.json` files into `c:\Users\Parth\OneDrive\Documents\AIML Project\results\`.
4. Run locally:
   ```bash
   uv run python evaluate.py
   uv run python compare.py
   uv run python app.py
   ```
All four models will now be evaluated and compared seamlessly!